# 👥 Module 7, 8 & 9: User-Based CF, Ranking, and the Cold-Start Problem
**Movie Recommendation System Using Machine Learning**

---

### 🎯 Objectives
1. Understand **User-to-User Collaborative Filtering** (finding peer neighbors).
2. Compare Item-Based CF vs User-Based CF (strengths, weaknesses, catalog coverage).
3. Understand the **Cold-Start Problem** (new users with zero ratings).
4. Implement the **IMDb Bayesian Weighted Rating** fallback strictly on training data.


In [ ]:
import os
import sys
import pandas as pd
import numpy as np

sys.path.insert(0, "..")
from src.recommender import MovieRecommender

model = MovieRecommender.load("../models/recommender_model.joblib")
print("Model ready!")


## 1. Finding Similar Users (Nearest Neighbors)
We compute similarity across users based on their rating overlap:
$$\text{sim}(u, v) = \frac{\mathbf{r}_u \cdot \mathbf{r}_v}{\|\mathbf{r}_u\| \|\mathbf{r}_v\|}$$


In [ ]:
from src.similarity import get_top_similar_users

target_user = 1
similar_users = get_top_similar_users(
    user_id=target_user,
    user_similarity=model.user_similarity,
    user_to_idx=model.user_to_idx,
    idx_to_user=model.idx_to_user,
    top_k=5
)

print(f"Top 5 users most similar to User {target_user}:")
for u, s in similar_users:
    print(f"  • User {u}: Similarity = {s:.4f}")


## 2. Generating User-Based Recommendations

In [ ]:
user_recs = model.recommend_user_based(user_id=1, top_k=5)
display(pd.DataFrame(user_recs))


## 3. The Cold-Start Problem & Bayesian Popularity Fallback

### What is the Cold-Start Problem?
When a new user joins, they have rated zero movies.
The similarity dot products are zero, so collaborative filtering fails!

### Solution: IMDb Bayesian Weighted Rating (WR)
$$\text{WR} = \left(\frac{v}{v + m}\right) R + \left(\frac{m}{v + m}\right) C$$
- $v$: Number of ratings the movie received in training data
- $m$: Minimum ratings threshold (confidence parameter)
- $R$: Average rating of the movie
- $C$: Mean rating across the entire training catalog


In [ ]:
# Test brand new user with ID 88888
new_user_id = 88888
cold_start_recs = model.recommend(user_id=new_user_id, top_k=5)
print(f"Recommendations for Cold-Start User {new_user_id}:")
display(pd.DataFrame(cold_start_recs))


## 4. Beginner Exercises
1. Compare the top 5 recommendations for User 1 between Item-Based CF and User-Based CF. Do they overlap?
2. Why is the Bayesian Weighted Rating superior to simple average rating for ranking popular movies?
